# Sycophancy Causal Effect — Pilot (Multi-Family)

Pilot run for project P9: *Measuring total causal effects of instruction tuning on LLM behaviour*, with **sycophancy** as the outcome variable, replicated across **three model families**: Qwen, Llama, and Gemma.

**Design**:
- Treatment: instruction tuning (base vs. instruct version of each family)
- Outcome: P(agree) with a user-stated claim, via next-token logits over `A`/`B`
- Moderator: 4 levels of premise strength (L0 neutral, L1 weak, L2 medium, L3 strong)
- Cross-family replication: 3 model families with different alignment recipes
- Source: TruthfulQA (`generation` config)

**VRAM strategy**: one (base, instruct) pair loaded at a time; `unload()` between families. Max ~8 GB used at any time on a 15 GB T4.

**Pilot scope**: 30 examples × 4 levels × 6 models (3 families × 2 roles) = **720 inference calls**.

## 1. Environment

In [9]:
# Verify GPU
!nvidia-smi

Fri May  8 22:47:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             30W /   70W |    6105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# Mount Drive (model cache + token storage)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Clone the project repo (or pull updates)
import os

REPO_URL = "https://github.com/SamueleSamonini/sycophancy-causal-effect.git"
REPO_DIR = "/content/sycophancy-causal-effect"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

print(f"Repo at {REPO_DIR}")

Already up to date.
Repo at /content/sycophancy-causal-effect


In [12]:
# Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

In [13]:
# Add repo to Python path so we can import src.*
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Standard libraries
import json
import torch
import pandas as pd
from tqdm import tqdm

# Project modules
from src.data.dataset_builder import (
    SycophancyDataset,
    extract_qa_triple,
    build_prompts,
    LEVEL_NAMES,
)
from src.models.inference import ModelScorer

print("All imports successful.")

All imports successful.


In [14]:
# Path setup: cache and results both on Drive
DRIVE_DIR   = "/content/drive/MyDrive/sycophancy-causal-effect"
CACHE_DIR   = f"{DRIVE_DIR}/cache"
RESULTS_DIR = f"{DRIVE_DIR}/results"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Cache:   {CACHE_DIR}")
print(f"Results: {RESULTS_DIR}")

Cache:   /content/drive/MyDrive/sycophancy-causal-effect/cache
Results: /content/drive/MyDrive/sycophancy-causal-effect/results


## 2. HuggingFace Authentication

Llama and Gemma are gated models that require accepting their license on HuggingFace
and authenticating with a personal access token (Read scope is sufficient).

The token is stored once in a private file on Drive and reused across sessions.

In [15]:
# Load HF token from Drive (or prompt for it the first time)
from huggingface_hub import login

HF_TOKEN_FILE = f"{DRIVE_DIR}/.hf_token"

if os.path.exists(HF_TOKEN_FILE):
    with open(HF_TOKEN_FILE) as f:
        HF_TOKEN = f.read().strip()
    print(f"HF token loaded from {HF_TOKEN_FILE}")
else:
    import getpass
    HF_TOKEN = getpass.getpass("Enter HuggingFace token (will be saved to Drive): ")
    with open(HF_TOKEN_FILE, "w") as f:
        f.write(HF_TOKEN)
    os.chmod(HF_TOKEN_FILE, 0o600)
    print(f"HF token saved to {HF_TOKEN_FILE}")

# Authenticate for the rest of the session: all from_pretrained calls will use this token
login(token=HF_TOKEN)

HF token loaded from /content/drive/MyDrive/sycophancy-causal-effect/.hf_token


## 3. Model Families

Three families with different instruction tuning recipes. Each row defines a (base, instruct) pair.

In [16]:
MODEL_FAMILIES = [
    {
        "name":     "Qwen2.5-1.5B",
        "base":     "Qwen/Qwen2.5-1.5B",
        "instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    },
    {
        "name":     "Llama-3.2-1B",
        "base":     "meta-llama/Llama-3.2-1B",
        "instruct": "meta-llama/Llama-3.2-1B-Instruct",
    },
    {
        "name":     "Gemma-2-2B",
        "base":     "google/gemma-2-2b",
        "instruct": "google/gemma-2-2b-it",
    },
]

print("Configured families:")
for f in MODEL_FAMILIES:
    print(f"  {f['name']:18s}  base={f['base']:40s}  instruct={f['instruct']}")

Configured families:
  Qwen2.5-1.5B        base=Qwen/Qwen2.5-1.5B                         instruct=Qwen/Qwen2.5-1.5B-Instruct
  Llama-3.2-1B        base=meta-llama/Llama-3.2-1B                   instruct=meta-llama/Llama-3.2-1B-Instruct
  Gemma-2-2B          base=google/gemma-2-2b                         instruct=google/gemma-2-2b-it


## 4. Dataset

Same 30 examples used across all families (fixed seed for reproducibility).

In [ ]:
dataset = SycophancyDataset(cache_dir=CACHE_DIR)
print(f"Valid examples available: {len(dataset)}")

PILOT_SIZE = 30
pilot_examples = dataset.sample(n=PILOT_SIZE, seed=42)
print(f"Pilot sample: {len(pilot_examples)} examples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Valid examples available: 817
Pilot sample: 30 examples


: 

## 5. Inference Across All Families

For each family: load (base, instruct), score all prompts, unload before moving to the next family.

If a family fails to load (e.g., gated license not yet granted), it is skipped with a warning
and the run continues with the remaining families.

In [18]:
records = []
skipped_families = []

for family in MODEL_FAMILIES:
    print(f"\n=== {family['name']} ===")
    
    # Try to load both versions; skip the family if either fails
    try:
        base = ModelScorer.load(family["base"], cache_dir=CACHE_DIR)
        print(f"  Base loaded:     VRAM = {torch.cuda.memory_allocated()/1e9:.2f} GB")
        
        instruct = ModelScorer.load(family["instruct"], cache_dir=CACHE_DIR)
        print(f"  Instruct loaded: VRAM = {torch.cuda.memory_allocated()/1e9:.2f} GB")
    except Exception as e:
        print(f"  Failed to load family '{family['name']}': {e}")
        skipped_families.append(family["name"])
        # Try to free anything that may have been partially loaded
        try: base.unload()
        except: pass
        try: instruct.unload()
        except: pass
        continue
    
    # Score all (example, level) combinations on both base and instruct
    role_scorers = [("base", base), ("instruct", instruct)]
    
    for ex_idx, ex in enumerate(tqdm(pilot_examples, desc=f"  Inference")):
        triple = extract_qa_triple(ex)
        prompts = build_prompts(triple)
        
        for level, prompt in prompts.items():
            for role, scorer in role_scorers:
                result = scorer.score_agreement(prompt)
                records.append({
                    "example_id":     ex_idx,
                    "question":       triple["question"],
                    "correct_answer": triple["correct_answer"],
                    "wrong_answer":   triple["wrong_answer"],
                    "level":          level,
                    "family":         family["name"],
                    "role":           role,
                    "model":          scorer.name,
                    "p_agree":        result["p_agree"],
                    "p_disagree":     result["p_disagree"],
                })
    
    # Free VRAM before the next family
    base.unload()
    instruct.unload()
    print(f"  Unloaded.        VRAM = {torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"\n=== Done ===")
print(f"Total records:     {len(records)}")
print(f"Skipped families:  {skipped_families if skipped_families else 'none'}")

df = pd.DataFrame(records)
df.head()


=== Qwen2.5-1.5B ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Base loaded:     VRAM = 3.09 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

: 

: 

## 6. Save and Aggregate

In [ ]:
csv_path  = f"{RESULTS_DIR}/pilot_multifamily_n{PILOT_SIZE}.csv"
json_path = f"{RESULTS_DIR}/pilot_multifamily_n{PILOT_SIZE}.json"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)

print(f"Saved CSV:  {csv_path}")
print(f"Saved JSON: {json_path}")
print()
print("Download both files from Drive and place them under `results/` in your local repo, then push via GitHub Desktop.")

In [ ]:
# Aggregate: mean P(agree) per (family, role, level)
agg = (
    df.groupby(["family", "role", "level"])["p_agree"]
      .agg(["mean", "std", "count"])
      .round(3)
)
print("=== Mean P(agree) per (family, role, level) ===")
print(agg)

In [ ]:
# Causal effect view: ATE_family,level = mean(p_agree | instruct) - mean(p_agree | base)
pivot = (
    df.groupby(["family", "level", "role"])["p_agree"]
      .mean()
      .unstack("role")
)
pivot["ATE"] = pivot["instruct"] - pivot["base"]
print("=== Treatment effect (instruct - base) per (family, level) ===")
print(pivot.round(3))